In [1]:
import sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
%reload_ext watermark
%watermark -a "Matheus dos Anjos" --iversion

Author: Matheus dos Anjos

sklearn: 1.5.1
pandas : 2.3.1



In [3]:
df = pd.read_csv('C:/Users/conta/OneDrive/Desktop/Projetos-/Projetos Python/Projetos de Machine Learning/dataset.csv')

In [4]:
df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,JobInvolvement,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Employee Source,AgeStartedWorking
0,41,Voluntary Resignation,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,...,0,8,0,1,6,4,0,5,Referral,33
1,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Human Resources,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29
2,41,Voluntary Resignation,Travel_Rarely,Sales,1,2,Life Sciences,2,Female,3,...,0,8,0,1,6,4,0,5,Referral,33
3,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Marketing,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29
4,37,Voluntary Resignation,Travel_Rarely,Human Resources,6,4,Human Resources,1,Female,3,...,0,8,0,1,6,4,0,5,Referral,29


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23058 entries, 0 to 23057
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       23058 non-null  int64 
 1   Attrition                 23058 non-null  object
 2   BusinessTravel            23058 non-null  object
 3   Department                23058 non-null  object
 4   DistanceFromHome          23058 non-null  int64 
 5   Education                 23058 non-null  int64 
 6   EducationField            23058 non-null  object
 7   EnvironmentSatisfaction   23058 non-null  int64 
 8   Gender                    23058 non-null  object
 9   JobInvolvement            23058 non-null  int64 
 10  JobLevel                  23058 non-null  int64 
 11  JobRole                   23058 non-null  object
 12  JobSatisfaction           23058 non-null  int64 
 13  MaritalStatus             23058 non-null  object
 14  MonthlyIncome         

In [6]:
df['Attrition'].value_counts()

Attrition
Current employee         19370
Voluntary Resignation     3601
Termination                 87
Name: count, dtype: int64

In [7]:
df = df[df['Attrition'].isin(['Current employee', 'Voluntary Resignation'])]

In [8]:
df['Attrition'].value_counts()

Attrition
Current employee         19370
Voluntary Resignation     3601
Name: count, dtype: int64

In [9]:
df['Attrition'] = df['Attrition'].apply(lambda x: 1 if x == 'Voluntary Resignation' else 0)

In [10]:
df['Attrition'].value_counts()

Attrition
0    19370
1     3601
Name: count, dtype: int64

In [11]:
X = df.drop('Attrition', axis = 1)
y = df['Attrition']

In [12]:
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state = 42)

In [13]:
cat_features = X.select_dtypes(include = ['object']).columns.tolist()
num_featrues = X.select_dtypes(include = ['int64', 'float64']).columns.tolist()

**Processamento automático**

In [20]:
numeric_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'median')),
    ('scaler', StandardScaler())])

**Processamento automático**

In [21]:
categorical_transformer = Pipeline(steps = [
    ('imputer', SimpleImputer(strategy = 'constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

**Processamento automático**

In [23]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, num_featrues),
        ('cat', categorical_transformer, cat_features)        
    ]
)

In [25]:
modelo = Pipeline(steps = [('preprocessor', preprocessor),
                   ('classifier', LogisticRegression(max_iter = 1000))])

In [26]:
modelo.fit(X_treino, y_treino) 

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'DistanceFromHome',
                                                   'Education',
                                                   'EnvironmentSatisfaction',
                                                   'JobInvolvement', 'JobLevel',
                                                   'JobSatisfaction',
                                                   'MonthlyIncome',
                                                   'NumCompaniesWorked',
                                                   'PercentSalaryHike',
                                                   'PerformanceRating',
                                                   '...
                                                   'YearsWithCurrManager',
                                                   'AgeStartedWorking ']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['BusinessTravel',
                                                   'Department',
                                                   'EducationField', 'Gender',
                                                   'JobRole', 'MaritalStatus',
                                                   'OverTime',
                                                   'Employee Source'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [29]:
y_pred = modelo.predict(X_teste)

In [30]:
acuracy = accuracy_score(y_teste, y_pred)

In [31]:
print(f'Acurácia: {acuracy:.2f}')

Acurácia: 0.86


In [38]:
coefficients = modelo.named_steps['classifier'].coef_[0]

In [33]:
feature_names = num_featrues + list(modelo.named_steps['preprocessor'].transformers_[1][1].named_steps['onehot'].get_feature_names_out(cat_features))

In [34]:
feature_names

['Age',
 'DistanceFromHome',
 'Education',
 'EnvironmentSatisfaction',
 'JobInvolvement',
 'JobLevel',
 'JobSatisfaction',
 'MonthlyIncome',
 'NumCompaniesWorked',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager',
 'AgeStartedWorking ',
 'BusinessTravel_Non-Travel',
 'BusinessTravel_Travel_Frequently',
 'BusinessTravel_Travel_Rarely',
 'Department_Human Resources',
 'Department_Research & Development',
 'Department_Sales',
 'EducationField_Human Resources',
 'EducationField_Life Sciences',
 'EducationField_Marketing',
 'EducationField_Medical',
 'EducationField_Other',
 'EducationField_Technical Degree',
 'Gender_Female',
 'Gender_Male',
 'JobRole_Healthcare Representative',
 'JobRole_Human Resources',
 'JobRole_Laboratory Technician',
 'JobRole_Manager',
 'JobRole_Manufacturing Dir

In [40]:
coeff_df = pd.DataFrame({'Attributo': feature_names, 'Coeficiente': coefficients}).sort_values(by = 'Coeficiente', ascending=False)

In [41]:
coeff_df.head(10)

,Attributo,Coeficiente
22,BusinessTravel_Travel_Frequently,0.494839
32,EducationField_Technical Degree,0.275768
56,Employee Source_Referral,0.257281
46,MaritalStatus_Single,0.213240
37,JobRole_Laboratory Technician,0.213197
48,OverTime_Yes,0.183383
1,DistanceFromHome,0.160642
18,YearsSinceLastPromotion,0.157113
43,JobRole_Sales Representative,0.126106
16,YearsAtCompany,0.105078


# Análise dos Resultados

**BusinessTravel_Travel_Frequently (0.494839)**: Funcionários que viajam frequentemente a negócios têm maior
probabilidade de se demitir.

**EducationField_Technical Degree (0.275768)**: Funcionários com diplomas tem uma probabilidade maior de se demitir. Maior chance de receber
propostas mais vantajosas por ser mais qualificado.

**Employee Source_Referral (0.257281)**: Funcionários que foram indicados tem maior probabilidade de se demitir.

**MaritalStatus_Single (0.213240)**: Funcionários solteiros tem mais chance de se demitir. Talves por terem mais flexibilidade, menos responsabilidade.

**JobRole_Laboratory Technician (0.213197)**: Técnicos de laboratório tem maior probabilidade de se demitir voluntariamente.